
# Stage1/Stage2 Experiment Analysis and Intermediate Visualization

This notebook does four things:
1. Parse `logs/` for Stage1/Stage2 experiment metadata and training curves.
2. Plot convergence, final loss, and throughput trends across runs.
3. Visualize intermediate outputs (input/reconstruction/importance/level-map/code distribution) from saved checkpoints.
4. Compare baseline vs importance-aware compression with proxy detection scores on a small validation subset.

Note:
- `teacher_drop` here is from the local proxy teacher adapter, not true PointPillars mAP.
- This notebook is designed to run inside the project root or inside `notebooks/`.


In [ ]:

import csv
import math
import re
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt


def find_repo_root(start: Path = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / "src").exists() and (p / "logs").exists():
            return p
    raise RuntimeError("Could not find repo root containing src/ and logs/.")


ROOT = find_repo_root()
NOTEBOOK_DIR = ROOT / "notebooks"
LOG_DIR = ROOT / "logs"
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT: {ROOT}")
print(f"LOG_DIR: {LOG_DIR}")


In [ ]:
# Parse .out and .err logs into summary structures + csv artifacts.


def parse_float(text, default=np.nan):
    try:
        return float(text)
    except Exception:
        return default


def to_seconds(ts: str) -> float:
    parts = [int(p) for p in ts.split(":")]
    if len(parts) == 3:
        h, m, s = parts
        return float(h * 3600 + m * 60 + s)
    if len(parts) == 2:
        m, s = parts
        return float(m * 60 + s)
    return float(parts[0])


def parse_out_file(path: Path):
    text = path.read_text(errors="ignore")
    meta_keys = [
        "stage",
        "training_mode",
        "backbone",
        "quantizer_mode",
        "quant_bits",
        "teacher_backend",
        "run_id",
        "lr",
        "roi_levels",
        "bg_levels",
        "roi_target_mode",
        "loss_weights",
        "save_dir",
        "started_at",
    ]
    meta = {}
    for k in meta_keys:
        m = re.search(rf"^{k}:\s*(.+)$", text, re.M)
        if m:
            meta[k] = m.group(1).strip()

    epoch_losses = []
    for m in re.finditer(r"^Epoch\s+(\d+):\s+Loss\s+([0-9.]+)(?:\s+\||$)", text, re.M):
        epoch_losses.append((int(m.group(1)), float(m.group(2))))

    return meta, epoch_losses


def parse_err_timing(path: Path):
    if not path.exists():
        return {}
    text = path.read_text(errors="ignore")
    pat = re.compile(r"Epoch\s+(\d+):\s+100%\|[^\[]*\[([0-9:]+)<[^,]*,\s*([0-9.]+)it/s")
    hits = pat.findall(text)
    by_epoch = {}
    for ep, elapsed, itps in hits:
        by_epoch[int(ep)] = (to_seconds(elapsed), float(itps))
    return by_epoch


def stage_display(stage: str) -> str:
    stage = str(stage)
    if stage == "0":
        return "Uniform Baseline (ROI-unaware)"
    if stage == "1":
        return "Adaptive ROI Compression (No Distill)"
    if stage == "2":
        return "Adaptive ROI Compression + Distillation"
    return f"Unknown Stage ({stage})"


def model_display_name(meta: dict, lam_distill: float) -> str:
    stage = str(meta.get("stage", ""))
    backbone = str(meta.get("backbone", "?")).upper()
    lr = str(meta.get("lr", "?"))
    qmode = str(meta.get("quantizer_mode", "?")).lower()
    qbits = str(meta.get("quant_bits", "?"))

    if stage == "0":
        return f"Uniform baseline | {backbone} | {qmode} q{qbits}"
    if stage == "1":
        return f"Adaptive ROI (no distill) | {backbone} | lr={lr}"
    if stage == "2":
        if lam_distill == lam_distill:
            return f"Adaptive ROI + distill | {backbone} | ld={lam_distill:.1f}"
        return f"Adaptive ROI + distill | {backbone}"
    return f"{backbone} | stage={stage}"


ANALYSIS_ONLY_WITH_EPOCH = True

out_files = sorted(p for p in LOG_DIR.glob("*_r*.out") if "slurm_" not in p.name)
runs = []
curve_map = {}
skipped_no_epoch = []

for out_path in out_files:
    meta, epoch_losses = parse_out_file(out_path)
    curve_map[out_path.name] = epoch_losses

    if ANALYSIS_ONLY_WITH_EPOCH and not epoch_losses:
        skipped_no_epoch.append(out_path.name)
        continue

    first_loss = epoch_losses[0][1] if epoch_losses else np.nan
    final_loss = epoch_losses[-1][1] if epoch_losses else np.nan
    delta_loss = first_loss - final_loss if epoch_losses else np.nan
    rel_improve_pct = (delta_loss / first_loss * 100.0) if epoch_losses and first_loss != 0 else np.nan

    lw = meta.get("loss_weights", "")
    lam_recon = parse_float(re.search(r"recon=([0-9.eE+-]+)", lw).group(1)) if re.search(r"recon=([0-9.eE+-]+)", lw) else np.nan
    lam_rate = parse_float(re.search(r"rate=([0-9.eE+-]+)", lw).group(1)) if re.search(r"rate=([0-9.eE+-]+)", lw) else np.nan
    lam_distill = parse_float(re.search(r"distill=([0-9.eE+-]+)", lw).group(1)) if re.search(r"distill=([0-9.eE+-]+)", lw) else np.nan
    lam_importance = parse_float(re.search(r"importance=([0-9.eE+-]+)", lw).group(1)) if re.search(r"importance=([0-9.eE+-]+)", lw) else np.nan

    timing = parse_err_timing(LOG_DIR / out_path.name.replace(".out", ".err"))
    if timing:
        secs = [timing[e][0] for e in sorted(timing)]
        itps = [timing[e][1] for e in sorted(timing)]
        epoch_sec_mean = float(np.mean(secs))
        epoch_sec_last = float(secs[-1])
        it_s_mean = float(np.mean(itps))
        it_s_last = float(itps[-1])
    else:
        epoch_sec_mean = np.nan
        epoch_sec_last = np.nan
        it_s_mean = np.nan
        it_s_last = np.nan

    runs.append(
        {
            "file": out_path.name,
            "stage": str(meta.get("stage", "")),
            "stage_label": stage_display(meta.get("stage", "")),
            "training_mode": meta.get("training_mode", ""),
            "backbone": meta.get("backbone", ""),
            "quantizer_mode": meta.get("quantizer_mode", ""),
            "quant_bits": meta.get("quant_bits", ""),
            "teacher_backend": meta.get("teacher_backend", ""),
            "run_id": meta.get("run_id", ""),
            "lr": meta.get("lr", ""),
            "roi_target_mode": meta.get("roi_target_mode", ""),
            "lambda_recon": lam_recon,
            "lambda_rate": lam_rate,
            "lambda_distill": lam_distill,
            "lambda_importance": lam_importance,
            "epoch_count": len(epoch_losses),
            "first_loss": first_loss,
            "final_loss": final_loss,
            "delta_loss": delta_loss,
            "rel_improve_pct": rel_improve_pct,
            "epoch_sec_mean": epoch_sec_mean,
            "epoch_sec_last": epoch_sec_last,
            "it_s_mean": it_s_mean,
            "it_s_last": it_s_last,
            "save_dir": meta.get("save_dir", ""),
            "started_at": meta.get("started_at", ""),
            "model_label": model_display_name(meta, lam_distill),
        }
    )

# stable sort for plotting
runs = sorted(
    runs,
    key=lambda r: (
        int(r["stage"]) if str(r["stage"]).isdigit() else 99,
        str(r["backbone"]),
        float(r["lambda_distill"]) if r["lambda_distill"] == r["lambda_distill"] else -1.0,
        str(r["lr"]),
        str(r["run_id"]),
    ),
)

# Save summary csv
summary_csv = NOTEBOOK_DIR / "analysis_log_summary.csv"
with summary_csv.open("w", newline="", encoding="utf-8") as f:
    fieldnames = list(runs[0].keys()) if runs else []
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(runs)

# Save curve csv
curve_csv = NOTEBOOK_DIR / "analysis_epoch_curves.csv"
with curve_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["file", "epoch", "loss"])
    writer.writeheader()
    for file_name, pts in curve_map.items():
        for ep, loss in pts:
            writer.writerow({"file": file_name, "epoch": ep, "loss": loss})

print(f"Parsed runs for analysis: {len(runs)}")
print(f"Skipped logs without epoch lines: {len(skipped_no_epoch)}")
if skipped_no_epoch:
    print("Examples skipped:", skipped_no_epoch[:5])
print(f"Saved: {summary_csv}")
print(f"Saved: {curve_csv}")

for r in runs:
    print(
        f"{r['stage_label']:<38} | {r['model_label']:<52} | "
        f"first={r['first_loss']:.4f} final={r['final_loss']:.4f} | it/s={r['it_s_mean']:.2f}"
    )


In [ ]:
# Focused curves: show only yesterday's Stage0/Stage1 representative runs.

FOCUS_DATE = "2026-02-24"  # yesterday-night sweep date
FOCUS_BACKBONE = "resnet"
FOCUS_STAGES = {"0", "1"}
FOCUS_LR = {"1e-4", "0.0001"}


def is_focus_date(row: dict) -> bool:
    started = str(row.get("started_at", ""))
    if started.startswith(FOCUS_DATE):
        return True
    save_dir = str(row.get("save_dir", ""))
    file_name = str(row.get("file", ""))
    return ("260224_" in save_dir) or file_name.startswith("260224_")


def pick_best(rows_local):
    if not rows_local:
        return None
    return sorted(rows_local, key=lambda z: (z["final_loss"], z["file"]))[0]


focused_pool = [
    r for r in runs
    if r["stage"] in FOCUS_STAGES
    and str(r.get("backbone", "")).lower() == FOCUS_BACKBONE
    and is_focus_date(r)
    and np.isfinite(r.get("final_loss", np.nan))
]

stage0_reps = []
for q in ("4", "6", "8"):
    cand = [r for r in focused_pool if r["stage"] == "0" and str(r.get("quant_bits", "")) == q]
    best = pick_best(cand)
    if best is not None:
        stage0_reps.append(best)

stage1_reps = []
for roi_mode in ("nearest", "maxpool", "area"):
    cand = [
        r for r in focused_pool
        if r["stage"] == "1"
        and str(r.get("roi_target_mode", "")).lower() == roi_mode
        and str(r.get("lr", "")).lower() in FOCUS_LR
    ]
    if not cand:
        cand = [
            r for r in focused_pool
            if r["stage"] == "1" and str(r.get("roi_target_mode", "")).lower() == roi_mode
        ]
    best = pick_best(cand)
    if best is not None:
        stage1_reps.append(best)

focused_runs = stage0_reps + stage1_reps

print(f"Focused pool size: {len(focused_pool)}")
print(f"Selected representative runs: {len(focused_runs)}")
for r in focused_runs:
    if r["stage"] == "0":
        desc = f"q{r['quant_bits']}"
    else:
        desc = f"roi={r.get('roi_target_mode', '?')}, lr={r.get('lr', '?')}"
    print(
        f"- stage{r['stage']} | {desc} | final={r['final_loss']:.4f} | "
        f"run_id={r.get('run_id', '?')} | file={r['file']}"
    )

if not focused_runs:
    print("No focused runs found. Check date/backbone filters.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(18, 5), dpi=130)

    # Stage0 panel (uniform q4/q6/q8)
    ax0 = axes[0]
    for r in stage0_reps:
        pts = curve_map.get(r["file"], [])
        if not pts:
            continue
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        label = f"q{r['quant_bits']} (final={r['final_loss']:.3f})"
        ax0.plot(xs, ys, linewidth=2.0, label=label)
    ax0.set_title("Stage0 (Yesterday, ResNet)\nUniform q4/q6/q8 representatives")
    ax0.set_xlabel("Epoch")
    ax0.set_ylabel("Avg training loss")
    ax0.grid(alpha=0.3)
    ax0.legend(fontsize=9)

    # Stage1 panel (roi target mode comparison)
    ax1 = axes[1]
    for r in stage1_reps:
        pts = curve_map.get(r["file"], [])
        if not pts:
            continue
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        roi = str(r.get("roi_target_mode", "?")).lower()
        label = f"{roi} (final={r['final_loss']:.3f})"
        ax1.plot(xs, ys, linewidth=2.0, label=label)
    ax1.set_title("Stage1 (Yesterday, ResNet, lr=1e-4)\nROI target mode representatives")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Avg training loss")
    ax1.grid(alpha=0.3)
    ax1.legend(fontsize=9)

    plt.tight_layout()
    plt.show()




In [ ]:
# Focused summary bars: only selected yesterday Stage0/Stage1 representatives.

if "focused_runs" not in globals() or not focused_runs:
    print("Run Cell 3 first (focused representative selection).")
else:
    stage0_sel = [r for r in focused_runs if r["stage"] == "0"]
    stage1_sel = [r for r in focused_runs if r["stage"] == "1"]
    selected = stage0_sel + stage1_sel

    labels = []
    for r in selected:
        if r["stage"] == "0":
            labels.append(f"S0-q{r['quant_bits']}")
        else:
            labels.append(f"S1-{str(r.get('roi_target_mode', '?')).lower()}")

    final_losses = [r["final_loss"] for r in selected]
    improves = [r["rel_improve_pct"] for r in selected]
    epoch_secs = [r["epoch_sec_mean"] for r in selected]
    colors = ["tab:green" if r["stage"] == "0" else "tab:blue" for r in selected]

    x = np.arange(len(selected))

    fig, axes = plt.subplots(1, 3, figsize=(20, 5), dpi=130)

    axes[0].bar(x, final_losses, color=colors)
    axes[0].set_title("Final Loss (Representative Runs)\nLower is better")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=25, ha="right", fontsize=9)
    axes[0].grid(axis="y", alpha=0.3)

    axes[1].bar(x, improves, color=colors)
    axes[1].set_title("Relative Improvement (%)")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=25, ha="right", fontsize=9)
    axes[1].grid(axis="y", alpha=0.3)

    axes[2].scatter(epoch_secs, final_losses, c=colors, s=100)
    for i, lbl in enumerate(labels):
        axes[2].annotate(lbl, (epoch_secs[i], final_losses[i]), fontsize=8)
    axes[2].set_title("Mean Epoch Time vs Final Loss")
    axes[2].set_xlabel("Mean epoch time (sec)")
    axes[2].set_ylabel("Final loss")
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Meaningful anchor comparison requested: Stage0 vs Stage1 (yesterday only)
    if stage0_sel and stage1_sel:
        best_s0 = sorted(stage0_sel, key=lambda z: z["final_loss"])[0]
        best_s1 = sorted(stage1_sel, key=lambda z: z["final_loss"])[0]
        delta = best_s0["final_loss"] - best_s1["final_loss"]
        rel = (delta / best_s0["final_loss"] * 100.0) if best_s0["final_loss"] != 0 else np.nan

        print("Meaningful comparison (yesterday Stage0 vs Stage1):")
        print(
            f"- Best Stage0: q{best_s0['quant_bits']} | final_loss={best_s0['final_loss']:.4f} | "
            f"run_id={best_s0.get('run_id', '?')}"
        )
        print(
            f"- Best Stage1: roi={best_s1.get('roi_target_mode', '?')} | final_loss={best_s1['final_loss']:.4f} | "
            f"run_id={best_s1.get('run_id', '?')}"
        )
        print(f"- Stage1 gain over Stage0 (loss delta): {delta:.4f} ({rel:.2f}%)")





## Intermediate Outputs and Proxy Detection

The next cells require `torch` and project modules (`src/`).
If torch import fails in your current kernel, switch kernel/environment (e.g., `lidarcomp311`).


In [ ]:

TORCH_AVAILABLE = True
TORCH_IMPORT_ERROR = None

try:
    import torch
    import torch.nn.functional as F
except Exception as e:
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = e

if not TORCH_AVAILABLE:
    print("Torch unavailable in this kernel:", repr(TORCH_IMPORT_ERROR))
else:
    src_dir = ROOT / "src"
    if str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

    from dataset.semantickitti_loader import SemanticKittiDataset
    import models.compression  # noqa: F401 (registry side effect)
    import models.backbones  # noqa: F401 (registry side effect)
    from models.registry import MODELS
    from utils.teacher_adapter import TeacherAdapter, TeacherAdapterConfig

    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())


In [ ]:
if TORCH_AVAILABLE:
    def build_model(
        backbone: str,
        quantizer_mode: str = "adaptive",
        quant_bits: int = 8,
        roi_levels: int = 256,
        bg_levels: int = 16,
    ):
        backbone_config = {"name": backbone, "in_channels": 5}
        decoder_stages = 5 if backbone == "darknet" else 4
        if backbone == "darknet":
            backbone_config["layers"] = (1, 1, 2, 2, 1)
        else:
            backbone_config["latent_channels"] = 64
            backbone_config["num_stages"] = 4
            backbone_config["blocks_per_stage"] = 1

        quantizer_mode = str(quantizer_mode or "adaptive").lower()
        qcfg = {"mode": quantizer_mode, "use_ste": True}
        if quantizer_mode == "adaptive":
            qcfg.update({"roi_levels": int(roi_levels), "bg_levels": int(bg_levels)})
        elif quantizer_mode == "uniform":
            qcfg.update({"uniform_bits": int(quant_bits), "quant_bits": int(quant_bits)})
        else:
            raise ValueError(f"Unsupported quantizer_mode: {quantizer_mode}")

        config = {
            "name": "lidar_compression",
            "backbone_config": backbone_config,
            "quantizer_config": qcfg,
            "decoder_config": {"latent_channels": 64, "out_channels": 5, "num_stages": decoder_stages},
            "head_config": (
                None
                if quantizer_mode == "uniform"
                else {"hidden_channels": 32, "activation": "relu"}
            ),
        }
        return MODELS.build(config)


    def load_ckpt_flexible(model, ckpt_path: Path, device):
        payload = torch.load(str(ckpt_path), map_location=device)
        if isinstance(payload, dict):
            payload = payload.get("model_state_dict", payload.get("model_state", payload.get("state_dict", payload)))
        missing, unexpected = model.load_state_dict(payload, strict=False)
        return missing, unexpected


    @torch.no_grad()
    def run_forward(model, x, mode="native"):
        # x: [B,5,H,W]
        if mode in ("native", "adaptive"):
            recon, aux = model(x, noise_std=0.0, quantize=True)
            return recon, aux
        if mode == "uniform_mean":
            _, aux_adapt = model(x, noise_std=0.0, quantize=True)
            if "importance_map_pred" not in aux_adapt:
                raise ValueError("uniform_mean mode requires adaptive quantizer with importance head.")
            imp = aux_adapt["importance_map_pred"]
            imp_mean = imp.mean(dim=(-2, -1), keepdim=True)
            imp_const = imp_mean.expand_as(imp)
            recon, aux = model(x, noise_std=0.0, quantize=True, importance_map=imp_const)
            return recon, aux
        raise ValueError(f"Unknown mode: {mode}")
else:
    print("Skip helper definitions because torch is unavailable.")



In [ ]:
if TORCH_AVAILABLE:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def parse_int_safe(x, default):
        try:
            return int(float(x))
        except Exception:
            return int(default)

    def latest_checkpoint(save_dir: str):
        save_path = Path(save_dir)
        if not save_path.is_absolute():
            save_path = ROOT / save_path
        if not save_path.exists():
            return None
        cands = []
        for p in save_path.glob("model_epoch_*.pth"):
            m = re.search(r"model_epoch_(\d+)\.pth", p.name)
            if m:
                cands.append((int(m.group(1)), p))
        if not cands:
            return None
        cands.sort(key=lambda x: x[0])
        return cands[-1][1]

    def log_date_id(run_file: str) -> int:
        m = re.match(r"^(\d{6})_", str(run_file))
        return int(m.group(1)) if m else -1

    def roi_mode_score(mode: str) -> int:
        mode = str(mode or "").lower()
        if mode == "maxpool":
            return 3
        if mode == "area":
            return 2
        if mode == "nearest":
            return 1
        return 0

    valid_runs = [r for r in runs if np.isfinite(r["final_loss"]) and r["save_dir"]]

    def pick_best(stage=None, backbone=None, quantizer_mode=None, quant_bits=None, prefer_roi_mode=True):
        cand = valid_runs
        if stage is not None:
            cand = [r for r in cand if str(r["stage"]) == str(stage)]
        if backbone is not None:
            cand = [r for r in cand if str(r["backbone"]) == str(backbone)]
        if quantizer_mode is not None:
            cand = [r for r in cand if str(r.get("quantizer_mode", "")).lower() == str(quantizer_mode).lower()]
        if quant_bits is not None:
            cand = [r for r in cand if str(r.get("quant_bits", "")) == str(quant_bits)]

        if prefer_roi_mode:
            cand = sorted(
                cand,
                key=lambda r: (
                    -roi_mode_score(r.get("roi_target_mode", "")),
                    -log_date_id(r.get("file", "")),
                    r["final_loss"],
                ),
            )
        else:
            cand = sorted(cand, key=lambda r: (r["final_loss"], -log_date_id(r.get("file", ""))))

        return cand[0] if cand else None

    def pick_stage2_fix(scoregate: bool):
        cand = [
            r
            for r in valid_runs
            if str(r["stage"]) == "2"
            and str(r["backbone"]) == "resnet"
            and str(r.get("quantizer_mode", "")).lower() == "adaptive"
            and "distill_fix2_energy_pool_16x32" in str(r.get("file", "")).lower()
        ]
        if scoregate:
            cand = [r for r in cand if "scoregate015" in str(r.get("file", "")).lower()]
        else:
            cand = [r for r in cand if "scoregate015" not in str(r.get("file", "")).lower()]

        cand = sorted(cand, key=lambda r: (r["final_loss"], -log_date_id(r.get("file", ""))))
        return cand[0] if cand else None

    selected_specs = [
        ("S0 Uniform Baseline (ResNet, q8)", pick_best(stage="0", backbone="resnet", quantizer_mode="uniform", quant_bits="8", prefer_roi_mode=False)),
        ("S1 Adaptive ROI (ResNet, no distill)", pick_best(stage="1", backbone="resnet", quantizer_mode="adaptive")),
        ("S2 Distill-Fix (Energy Pool 16x32, no score-gate)", pick_stage2_fix(scoregate=False)),
        ("S2 Distill-Fix (Energy Pool 16x32, score-gate=0.15)", pick_stage2_fix(scoregate=True)),
    ]

    # remove None + duplicates by file path
    chosen = []
    seen_files = set()
    for alias, row in selected_specs:
        if row is None:
            continue
        if row["file"] in seen_files:
            continue
        seen_files.add(row["file"])
        chosen.append((alias, row))

    data_root = ROOT / "data/dataset/semantickitti/dataset/sequences"
    val_seq = "08"
    sample_index = 100

    dataset = SemanticKittiDataset(root_dir=str(data_root), sequences=[val_seq], return_roi_mask=True)
    sample_index = min(sample_index, len(dataset) - 1)
    data, valid_mask, roi_mask = dataset[sample_index]

    x = data.unsqueeze(0).to(device)
    valid_b = valid_mask.unsqueeze(0).to(device)
    roi_b = roi_mask.unsqueeze(0).to(device)

    outputs = {}
    selected_runs = []

    for alias, row in chosen:
        ckpt_path = latest_checkpoint(row["save_dir"])
        if ckpt_path is None:
            print(f"Skip {alias}: no model_epoch_*.pth in {row['save_dir']}")
            continue

        backbone = str(row.get("backbone", "darknet"))
        qmode = str(row.get("quantizer_mode", "adaptive") or "adaptive").lower()
        qbits = parse_int_safe(row.get("quant_bits", 8), 8)
        roi_levels = parse_int_safe(row.get("roi_levels", 256), 256)
        bg_levels = parse_int_safe(row.get("bg_levels", 16), 16)

        model = build_model(
            backbone,
            quantizer_mode=qmode,
            quant_bits=qbits,
            roi_levels=roi_levels,
            bg_levels=bg_levels,
        ).to(device)
        missing, unexpected = load_ckpt_flexible(model, ckpt_path, device)
        if missing:
            print(f"[{alias}] missing keys: {len(missing)}")
        if unexpected:
            print(f"[{alias}] unexpected keys: {len(unexpected)}")

        model.eval()
        recon_adapt, aux_adapt = run_forward(model, x, mode="native")

        recon_uniform = None
        aux_uniform = None
        if str(row["stage"]) == "2" and qmode == "adaptive":
            recon_uniform, aux_uniform = run_forward(model, x, mode="uniform_mean")

        outputs[alias] = {
            "recon_adapt": recon_adapt.detach().cpu(),
            "aux_adapt": {k: (v.detach().cpu() if torch.is_tensor(v) else v) for k, v in aux_adapt.items()},
            "recon_uniform": recon_uniform.detach().cpu() if recon_uniform is not None else None,
            "aux_uniform": {k: (v.detach().cpu() if torch.is_tensor(v) else v) for k, v in aux_uniform.items()} if aux_uniform is not None else None,
            "backbone": backbone,
            "meta": row,
            "ckpt_path": str(ckpt_path),
        }

        selected_runs.append(
            {
                "alias": alias,
                "backbone": backbone,
                "ckpt_path": str(ckpt_path),
                "meta": row,
            }
        )

    x_cpu = x.detach().cpu()
    valid_cpu = valid_b.detach().cpu()
    roi_cpu = roi_b.detach().cpu()

    print("Loaded outputs:", sorted(outputs.keys()))
    print("Sample tensor shape:", list(x_cpu.shape))
    print("Selected checkpoint set:")
    for s in selected_runs:
        m = s["meta"]
        print(
            f"- {s['alias']}: stage={m['stage']} ({m['stage_label']}), backbone={m['backbone']}, "
            f"quantizer={m.get('quantizer_mode', '?')}, roi_target={m.get('roi_target_mode', '?')}, "
            f"final_loss={m['final_loss']:.4f}, run_file={m['file']}"
        )
else:
    print("Skip sample inference because torch is unavailable.")



In [ ]:
if TORCH_AVAILABLE and outputs:
    from matplotlib.colors import ListedColormap
    import matplotlib.patches as mpatches

    # pick representative runs by stage
    stage1_candidates = [k for k, v in outputs.items() if str(v["meta"].get("stage", "")) == "1"]
    stage2_candidates = [k for k, v in outputs.items() if str(v["meta"].get("stage", "")) == "2"]

    def pick_with_preference(keys, token):
        for k in keys:
            if token.lower() in k.lower():
                return k
        return keys[0] if keys else None

    stage1_key = pick_with_preference(stage1_candidates, "resnet") or sorted(outputs.keys())[0]
    stage2_key = pick_with_preference(stage2_candidates, "score-gate") or pick_with_preference(stage2_candidates, "scoregate") or (stage2_candidates[0] if stage2_candidates else stage1_key)

    x0 = x_cpu[0]
    H, W = x0.shape[-2:]
    valid0 = valid_cpu[0].numpy() > 0.5
    roi0 = roi_cpu[0, 0].numpy() > 0.5

    range_in = x0[0].numpy()
    intensity_in = x0[1].numpy()

    rec1 = outputs[stage1_key]["recon_adapt"][0, 0].numpy()
    rec2 = outputs[stage2_key]["recon_adapt"][0, 0].numpy()
    err1 = np.abs(rec1 - range_in)
    err2 = np.abs(rec2 - range_in)

    imp2 = outputs[stage2_key]["aux_adapt"].get("importance_map_pred", None)
    if imp2 is None:
        raise RuntimeError(f"No importance_map_pred in selected run: {stage2_key}")
    imp2_up = F.interpolate(imp2, size=(H, W), mode="bilinear", align_corners=False)[0, 0].numpy()

    lvl2 = outputs[stage2_key]["aux_adapt"].get("level_map", None)
    if lvl2 is not None:
        lvl2_up = F.interpolate(lvl2, size=(H, W), mode="nearest")[0, 0].numpy()
    else:
        lvl2_up = np.zeros((H, W), dtype=np.float32)

    gt_roi = roi0 & valid0

    def roi_metrics_at_threshold(thr: float):
        pred_roi_local = (imp2_up >= thr) & valid0
        tp = pred_roi_local & gt_roi
        fp = pred_roi_local & (~gt_roi)
        fn = (~pred_roi_local) & gt_roi
        tp_n = int(tp.sum())
        fp_n = int(fp.sum())
        fn_n = int(fn.sum())
        precision = tp_n / max(tp_n + fp_n, 1)
        recall = tp_n / max(tp_n + fn_n, 1)
        iou = tp_n / max(tp_n + fp_n + fn_n, 1)
        return pred_roi_local, tp, fp, fn, tp_n, fp_n, fn_n, precision, recall, iou

    # fixed-threshold metric used in plots
    pred_thr = 0.50
    pred_roi, tp, fp, fn, tp_n, fp_n, fn_n, precision, recall, iou = roi_metrics_at_threshold(pred_thr)

    # threshold sweep for diagnosis
    thr_list = np.linspace(0.01, 0.99, 33)
    sweep = []
    for t in thr_list:
        _, _, _, _, tp_i, fp_i, fn_i, p_i, r_i, iou_i = roi_metrics_at_threshold(float(t))
        sweep.append({"thr": float(t), "tp": tp_i, "fp": fp_i, "fn": fn_i, "precision": p_i, "recall": r_i, "iou": iou_i})
    sweep_sorted = sorted(sweep, key=lambda z: z["iou"], reverse=True)

    def masked(arr):
        out = arr.copy().astype(np.float32)
        out[~valid0] = np.nan
        return out

    # robust ranges for visualization
    range_valid = range_in[valid0]
    inten_valid = intensity_in[valid0]
    range_vmin, range_vmax = np.percentile(range_valid, [2, 98]) if range_valid.size else (0.0, 1.0)
    inten_vmin, inten_vmax = np.percentile(inten_valid, [2, 98]) if inten_valid.size else (0.0, 1.0)
    err_vmax = np.percentile(err2[valid0], 98) if valid0.any() else float(np.nanmax(err2))

    # -----------------------------------------------------------------
    # A) Wide input/reconstruction views (single-sample diagnostic)
    # -----------------------------------------------------------------
    fig, axes = plt.subplots(7, 1, figsize=(26, 18), dpi=130)

    im = axes[0].imshow(masked(range_in), cmap="viridis", aspect="auto", vmin=range_vmin, vmax=range_vmax)
    axes[0].set_title("Input Range (single-frame diagnostic)", fontsize=13)
    plt.colorbar(im, ax=axes[0], fraction=0.012, pad=0.01)

    im = axes[1].imshow(masked(intensity_in), cmap="magma", aspect="auto", vmin=inten_vmin, vmax=inten_vmax)
    axes[1].set_title("Input Intensity (single-frame diagnostic)", fontsize=13)
    plt.colorbar(im, ax=axes[1], fraction=0.012, pad=0.01)

    im = axes[2].imshow(masked(gt_roi.astype(np.float32)), cmap="gray", aspect="auto", vmin=0, vmax=1)
    axes[2].set_title("GT ROI Mask (single frame)", fontsize=13)
    plt.colorbar(im, ax=axes[2], fraction=0.012, pad=0.01)

    im = axes[3].imshow(masked(rec1), cmap="viridis", aspect="auto", vmin=range_vmin, vmax=range_vmax)
    axes[3].set_title(f"Reconstructed Range - {stage1_key}", fontsize=13)
    plt.colorbar(im, ax=axes[3], fraction=0.012, pad=0.01)

    im = axes[4].imshow(masked(rec2), cmap="viridis", aspect="auto", vmin=range_vmin, vmax=range_vmax)
    axes[4].set_title(f"Reconstructed Range - {stage2_key}", fontsize=13)
    plt.colorbar(im, ax=axes[4], fraction=0.012, pad=0.01)

    im = axes[5].imshow(masked(err1), cmap="inferno", aspect="auto", vmin=0.0, vmax=err_vmax)
    axes[5].set_title(f"Absolute Error Map - {stage1_key}", fontsize=13)
    plt.colorbar(im, ax=axes[5], fraction=0.012, pad=0.01)

    im = axes[6].imshow(masked(err2), cmap="inferno", aspect="auto", vmin=0.0, vmax=err_vmax)
    axes[6].set_title(f"Absolute Error Map - {stage2_key}", fontsize=13)
    plt.colorbar(im, ax=axes[6], fraction=0.012, pad=0.01)

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()

    # -----------------------------------------------------------------
    # B) ROI / Importance diagnostic views (single frame)
    # -----------------------------------------------------------------
    fig, axes = plt.subplots(5, 1, figsize=(26, 14), dpi=130)

    im = axes[0].imshow(masked(imp2_up), cmap="plasma", aspect="auto", vmin=0.0, vmax=1.0)
    axes[0].set_title("Predicted Importance Map (upsampled)", fontsize=13)
    plt.colorbar(im, ax=axes[0], fraction=0.012, pad=0.01)

    im = axes[1].imshow(masked(pred_roi.astype(np.float32)), cmap="gray", aspect="auto", vmin=0, vmax=1)
    axes[1].set_title(f"Predicted ROI from Importance (threshold={pred_thr:.2f})", fontsize=13)
    plt.colorbar(im, ax=axes[1], fraction=0.012, pad=0.01)

    overlap = np.zeros((H, W), dtype=np.int32)
    overlap[tp] = 1
    overlap[fp] = 2
    overlap[fn] = 3
    overlap[~valid0] = 0

    cmap_overlap = ListedColormap(["black", "lime", "red", "gold"])
    axes[2].imshow(overlap, cmap=cmap_overlap, aspect="auto", vmin=0, vmax=3)
    axes[2].set_title("ROI Overlap (TP=green, FP=red, FN=yellow)", fontsize=13)

    handles = [
        mpatches.Patch(color="lime", label="TP"),
        mpatches.Patch(color="red", label="FP"),
        mpatches.Patch(color="gold", label="FN"),
    ]
    axes[2].legend(handles=handles, loc="upper right", ncol=3, framealpha=0.9)

    axes[3].imshow(masked(range_in), cmap="gray", aspect="auto", vmin=range_vmin, vmax=range_vmax)
    axes[3].contour(gt_roi.astype(np.float32), levels=[0.5], colors=["cyan"], linewidths=1.0)
    axes[3].contour(pred_roi.astype(np.float32), levels=[0.5], colors=["magenta"], linewidths=1.0)
    axes[3].set_title("Contour Overlay on Input Range (GT=cyan, Pred=magenta)", fontsize=13)

    im = axes[4].imshow(masked(lvl2_up), cmap="cividis", aspect="auto")
    axes[4].set_title("Adaptive Level Map (upsampled to input)", fontsize=13)
    plt.colorbar(im, ax=axes[4], fraction=0.012, pad=0.01)

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle(
        f"Single-frame ROI Diagnostic | IoU={iou:.4f}, Precision={precision:.4f}, Recall={recall:.4f}",
        fontsize=15,
        y=1.01,
    )
    plt.tight_layout()
    plt.show()

    # -----------------------------------------------------------------
    # C) Threshold sweep curve (to interpret why fixed threshold can fail)
    # -----------------------------------------------------------------
    thr_axis = [d["thr"] for d in sweep]
    iou_axis = [d["iou"] for d in sweep]
    p_axis = [d["precision"] for d in sweep]
    r_axis = [d["recall"] for d in sweep]

    fig, ax = plt.subplots(1, 1, figsize=(10, 4), dpi=130)
    ax.plot(thr_axis, iou_axis, label="IoU", linewidth=2)
    ax.plot(thr_axis, p_axis, label="Precision", linewidth=2)
    ax.plot(thr_axis, r_axis, label="Recall", linewidth=2)
    ax.axvline(pred_thr, color="black", linestyle="--", linewidth=1, label=f"fixed thr={pred_thr:.2f}")
    ax.set_title("ROI Metrics vs Importance Threshold (single frame)")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Metric")
    ax.grid(alpha=0.3)
    ax.set_ylim(-0.02, 1.02)
    ax.legend()
    plt.tight_layout()
    plt.show()

    imp_valid = imp2_up[valid0]
    print("[ROI/Importance Metrics - Single Frame Diagnostic]")
    print(f"- stage2 run: {stage2_key}")
    print(f"- stage1 reference: {stage1_key}")
    print(f"- threshold (fixed for plot): {pred_thr:.2f}")
    print(f"- TP={tp_n}, FP={fp_n}, FN={fn_n}")
    print(f"- IoU={iou:.6f}, Precision={precision:.6f}, Recall={recall:.6f}")
    print(
        f"- importance stats on valid pixels: min={imp_valid.min():.4f}, max={imp_valid.max():.4f}, "
        f"mean={imp_valid.mean():.4f}, p95={np.percentile(imp_valid,95):.4f}, p99={np.percentile(imp_valid,99):.4f}"
    )
    if float(imp_valid.max()) < pred_thr:
        print(
            f"- diagnostic: max importance ({float(imp_valid.max()):.4f}) < threshold ({pred_thr:.2f}), "
            "so predicted ROI is empty by definition at this threshold."
        )
    print("- top thresholds by IoU:")
    for item in sweep_sorted[:5]:
        print(
            f"  thr={item['thr']:.2f} | IoU={item['iou']:.4f} | "
            f"P={item['precision']:.4f} R={item['recall']:.4f} | TP={item['tp']} FP={item['fp']} FN={item['fn']}"
        )
else:
    print("Skip visualization: outputs not available.")




In [ ]:
if TORCH_AVAILABLE and outputs:
    if "stage2_key" not in globals() or stage2_key not in outputs:
        stage2_candidates = [k for k, v in outputs.items() if str(v["meta"].get("stage", "")) == "2"]
        stage2_key = stage2_candidates[0] if stage2_candidates else sorted(outputs.keys())[0]

    aux = outputs[stage2_key]["aux_adapt"]
    codes = aux["codes"][0]  # [C,h,w]
    lvl = aux["level_map"][0, 0]

    roi_lat = F.interpolate(roi_cpu, size=codes.shape[-2:], mode="nearest")[0, 0] > 0.5
    roi_lat_expand = roi_lat.unsqueeze(0).expand(codes.shape[0], -1, -1)

    roi_codes = codes[roi_lat_expand].numpy().reshape(-1)
    bg_codes = codes[~roi_lat_expand].numpy().reshape(-1)

    lvl_np = lvl.numpy()
    roi_lat_np = roi_lat.numpy()
    lvl_roi = lvl_np[roi_lat_np]
    lvl_bg = lvl_np[~roi_lat_np]

    fig, axes = plt.subplots(2, 2, figsize=(20, 10), dpi=130)

    axes[0, 0].hist(roi_codes, bins=60, alpha=0.70, label="ROI", color="tab:orange", density=True)
    axes[0, 0].hist(bg_codes, bins=60, alpha=0.55, label="BG", color="tab:blue", density=True)
    axes[0, 0].set_title(f"Quantized Code Distribution ({stage2_key}, adaptive)")
    axes[0, 0].set_xlabel("Code value")
    axes[0, 0].set_ylabel("Density")
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    axes[0, 1].hist(lvl_roi, bins=40, alpha=0.70, label="ROI", color="tab:orange", density=True)
    axes[0, 1].hist(lvl_bg, bins=40, alpha=0.55, label="BG", color="tab:blue", density=True)
    axes[0, 1].set_title("Level-map Distribution (ROI vs BG)")
    axes[0, 1].set_xlabel("Quantization levels")
    axes[0, 1].set_ylabel("Density")
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)

    im = axes[1, 0].imshow(lvl_np, cmap="viridis", aspect="auto")
    axes[1, 0].contour(roi_lat_np.astype(np.float32), levels=[0.5], colors=["white"], linewidths=0.8)
    axes[1, 0].set_title("Latent Level-map with ROI contour")
    axes[1, 0].set_xticks([])
    axes[1, 0].set_yticks([])
    plt.colorbar(im, ax=axes[1, 0], fraction=0.020, pad=0.01)

    summary_txt = (
        f"Code mean ROI/BG: {roi_codes.mean():.2f} / {bg_codes.mean():.2f}\n"
        f"Code std ROI/BG: {roi_codes.std():.2f} / {bg_codes.std():.2f}\n"
        f"Level mean ROI/BG: {lvl_roi.mean():.2f} / {lvl_bg.mean():.2f}\n"
        f"Level std ROI/BG: {lvl_roi.std():.2f} / {lvl_bg.std():.2f}"
    )
    axes[1, 1].axis("off")
    axes[1, 1].text(0.01, 0.95, summary_txt, va="top", ha="left", fontsize=12, family="monospace")
    axes[1, 1].set_title("ROI/BG Quantization Summary")

    plt.tight_layout()
    plt.show()
else:
    print("Skip code histogram: outputs not available.")

In [ ]:
# Small-subset comparison for proxy task metrics (16 frames).
# This is NOT final detector mAP; it is a fast diagnostic.

if TORCH_AVAILABLE:
    RECOMPUTE = True
    MAX_FRAMES = 16
    BATCH_SIZE = 2

    summary_path = NOTEBOOK_DIR / f"quick_eval_summary_{MAX_FRAMES}frames.csv"
    detail_path = NOTEBOOK_DIR / f"quick_eval_detail_{MAX_FRAMES}frames.csv"

    def read_csv_rows(path: Path):
        with path.open("r", encoding="utf-8") as f:
            return list(csv.DictReader(f))

    def write_csv_rows(path: Path, rows, fieldnames):
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader()
            w.writerows(rows)

    @torch.no_grad()
    def evaluate_run(model, loader, device, teacher, mode="native", quantizer_mode="adaptive", quant_bits=np.nan):
        model.eval()
        rows = []
        frame_idx = 0
        for data, valid_mask, roi_mask in loader:
            data = data.to(device)
            valid_mask = valid_mask.to(device)
            roi_mask = roi_mask.to(device)
            valid = valid_mask.unsqueeze(1)

            importance_override = None
            if mode == "uniform_mean":
                _, aux_tmp = model(data, noise_std=0.0, quantize=True)
                imp = aux_tmp.get("importance_map_pred", None)
                if imp is None:
                    raise RuntimeError("uniform_mean ablation requires adaptive model with importance_map_pred")
                imp_mean = imp.mean(dim=(-2, -1), keepdim=True)
                importance_override = imp_mean.expand_as(imp)

            recon, aux = model(data, noise_std=0.0, quantize=True, importance_map=importance_override)

            sq = (recon - data).pow(2)
            roi = roi_mask
            bg = 1.0 - roi

            all_w = valid
            roi_w = valid * roi
            bg_w = valid * bg

            all_mse = (sq * all_w).sum(dim=(1, 2, 3)) / all_w.sum(dim=(1, 2, 3)).clamp(min=1.0)
            roi_mse = (sq * roi_w).sum(dim=(1, 2, 3)) / roi_w.sum(dim=(1, 2, 3)).clamp(min=1.0)
            bg_mse = (sq * bg_w).sum(dim=(1, 2, 3)) / bg_w.sum(dim=(1, 2, 3)).clamp(min=1.0)

            if "level_map" in aux:
                rate_proxy = aux["level_map"].mean(dim=(1, 2, 3))
            elif str(quantizer_mode).lower() == "uniform" and quant_bits == quant_bits:
                rate_proxy = torch.full_like(all_mse, float(quant_bits))
            else:
                rate_proxy = torch.full_like(all_mse, float("nan"))

            if "importance_map_pred" in aux:
                imp = aux["importance_map_pred"]
                roi_ds = F.interpolate(roi, size=imp.shape[-2:], mode="nearest")
                bg_ds = 1.0 - roi_ds
                valid_ds = F.interpolate(valid, size=imp.shape[-2:], mode="nearest")
                roi_imp = (imp * roi_ds * valid_ds).sum(dim=(1, 2, 3)) / (roi_ds * valid_ds).sum(dim=(1, 2, 3)).clamp(min=1.0)
                bg_imp = (imp * bg_ds * valid_ds).sum(dim=(1, 2, 3)) / (bg_ds * valid_ds).sum(dim=(1, 2, 3)).clamp(min=1.0)
                imp_gap = roi_imp - bg_imp
            else:
                imp_gap = torch.full_like(all_mse, float("nan"))

            orig = teacher.infer(data, valid_mask=valid)
            rec = teacher.infer(recon, valid_mask=valid)
            drop = orig["score"] - rec["score"]

            for i in range(data.shape[0]):
                rows.append(
                    {
                        "frame_idx": frame_idx,
                        "all_mse": float(all_mse[i].cpu()),
                        "roi_mse": float(roi_mse[i].cpu()),
                        "bg_mse": float(bg_mse[i].cpu()),
                        "rate_proxy": float(rate_proxy[i].cpu()),
                        "teacher_drop": float(drop[i].cpu()),
                        "imp_roi_gap": float(imp_gap[i].cpu()),
                    }
                )
                frame_idx += 1

        return rows

    def mean_of(rows, key):
        vals = [float(r[key]) for r in rows if not np.isnan(float(r[key]))]
        return float(np.mean(vals)) if vals else float("nan")

    if RECOMPUTE or (not summary_path.exists()) or (not detail_path.exists()):
        ds = SemanticKittiDataset(root_dir=str(ROOT / "data/dataset/semantickitti/dataset/sequences"), sequences=["08"], return_roi_mask=True)
        ds = torch.utils.data.Subset(ds, list(range(min(MAX_FRAMES, len(ds)))))
        loader = torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        teacher = TeacherAdapter(TeacherAdapterConfig(backend="proxy", device="cuda" if torch.cuda.is_available() else "cpu"))

        runs_to_eval = []
        for s in selected_runs:
            stg = str(s["meta"].get("stage", ""))
            if stg not in ("0", "1", "2"):
                continue
            qmode = str(s["meta"].get("quantizer_mode", "adaptive") or "adaptive").lower()
            qbits = parse_int_safe(s["meta"].get("quant_bits", 8), 8)
            runs_to_eval.append(
                (
                    s["alias"],
                    s["backbone"],
                    Path(s["ckpt_path"]),
                    "native",
                    stg,
                    float(s["meta"].get("lambda_distill", np.nan)),
                    qmode,
                    qbits,
                )
            )

        # Add one uniform-mean ablation for the first stage2 adaptive checkpoint
        stage2_adaptive = [r for r in runs_to_eval if r[4] == "2" and r[6] == "adaptive"]
        if stage2_adaptive:
            base = stage2_adaptive[0]
            runs_to_eval.append((f"{base[0]} | uniform-mean ablation", base[1], base[2], "uniform_mean", base[4], base[5], base[6], base[7]))

        device_eval = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        summary_rows = []
        detail_rows = []

        for run_name, backbone, ckpt_path, mode, stage_id, ld, qmode, qbits in runs_to_eval:
            if not ckpt_path.exists():
                print("Missing checkpoint, skip:", ckpt_path)
                continue
            model = build_model(backbone, quantizer_mode=qmode, quant_bits=qbits).to(device_eval)
            load_ckpt_flexible(model, ckpt_path, device_eval)
            rows = evaluate_run(model, loader, device_eval, teacher, mode=mode, quantizer_mode=qmode, quant_bits=qbits)

            for r in rows:
                r["run"] = run_name
                r["mode"] = mode
                r["backbone"] = backbone
                r["stage"] = stage_id
                r["quantizer_mode"] = qmode
                r["quant_bits"] = qbits
                detail_rows.append(r)

            summary_rows.append(
                {
                    "run": run_name,
                    "stage": stage_id,
                    "mode": mode,
                    "backbone": backbone,
                    "quantizer_mode": qmode,
                    "quant_bits": qbits,
                    "lambda_distill": ld,
                    "frames": len(rows),
                    "all_mse_mean": mean_of(rows, "all_mse"),
                    "roi_mse_mean": mean_of(rows, "roi_mse"),
                    "bg_mse_mean": mean_of(rows, "bg_mse"),
                    "rate_proxy_mean": mean_of(rows, "rate_proxy"),
                    "teacher_drop_mean": mean_of(rows, "teacher_drop"),
                    "imp_roi_gap_mean": mean_of(rows, "imp_roi_gap"),
                }
            )

        write_csv_rows(
            summary_path,
            summary_rows,
            ["run", "stage", "mode", "backbone", "quantizer_mode", "quant_bits", "lambda_distill", "frames", "all_mse_mean", "roi_mse_mean", "bg_mse_mean", "rate_proxy_mean", "teacher_drop_mean", "imp_roi_gap_mean"],
        )
        write_csv_rows(
            detail_path,
            detail_rows,
            ["run", "stage", "mode", "backbone", "quantizer_mode", "quant_bits", "frame_idx", "all_mse", "roi_mse", "bg_mse", "rate_proxy", "teacher_drop", "imp_roi_gap"],
        )

    summary_rows = read_csv_rows(summary_path)
    detail_rows = read_csv_rows(detail_path)

    print(f"Loaded quick-eval summary rows: {len(summary_rows)}")
    for r in summary_rows:
        print(
            r["run"],
            "| stage=", r["stage"],
            "| qmode=", r.get("quantizer_mode", "?"),
            "| all_mse=", f"{float(r['all_mse_mean']):.4f}",
            "| roi_mse=", f"{float(r['roi_mse_mean']):.4f}",
            "| rate=", f"{float(r['rate_proxy_mean']):.4f}",
            "| drop=", f"{float(r['teacher_drop_mean']):.6f}",
            "| imp_gap=", f"{float(r['imp_roi_gap_mean']):.6f}",
        )
else:
    print("Skip quick-eval: torch unavailable.")



In [ ]:
if TORCH_AVAILABLE and summary_rows:
    rows_local = sorted(summary_rows, key=lambda r: (int(r.get("stage", "9")), r["run"]))

    names = [r["run"] for r in rows_local]
    all_mse = [float(r["all_mse_mean"]) for r in rows_local]
    roi_mse = [float(r["roi_mse_mean"]) for r in rows_local]
    rate = [float(r["rate_proxy_mean"]) for r in rows_local]
    drop = [float(r["teacher_drop_mean"]) for r in rows_local]
    imp_gap = [float(r["imp_roi_gap_mean"]) for r in rows_local]
    stages = [str(r.get("stage", "")) for r in rows_local]

    x = np.arange(len(names))
    colors = ["tab:green" if s == "0" else ("tab:blue" if s == "1" else "tab:orange") for s in stages]

    fig, axes = plt.subplots(1, 4, figsize=(28, 6), dpi=120)

    axes[0].bar(x, all_mse, color=colors, alpha=0.85, label="all_mse")
    axes[0].bar(x, roi_mse, color="tab:red", alpha=0.45, label="roi_mse")
    axes[0].set_title("Reconstruction Error (overall vs ROI) | lower is better")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(names, rotation=40, ha="right", fontsize=8)
    axes[0].legend()
    axes[0].grid(axis="y", alpha=0.3)

    axes[1].bar(x, rate, color=colors)
    axes[1].set_title("Rate Proxy (Stage0 uses quant_bits const; Stage1/2 use mean level_map)")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(names, rotation=40, ha="right", fontsize=8)
    axes[1].grid(axis="y", alpha=0.3)

    axes[2].bar(x, drop, color=colors)
    axes[2].set_title("Proxy Teacher Score Drop (orig - recon) | closer to 0 is better")
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(names, rotation=40, ha="right", fontsize=8)
    axes[2].axhline(0.0, color="black", linewidth=1)
    axes[2].grid(axis="y", alpha=0.3)

    axes[3].bar(x, imp_gap, color=colors)
    axes[3].set_title("Importance Gap (ROI mean - BG mean) | positive is desired")
    axes[3].set_xticks(x)
    axes[3].set_xticklabels(names, rotation=40, ha="right", fontsize=8)
    axes[3].axhline(0.0, color="black", linewidth=1)
    axes[3].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Adaptive vs uniform-mean ablation scatter (if available)
    uniform_ablation = [r for r in detail_rows if "uniform-mean ablation" in r["run"]]
    if uniform_ablation:
        base_names = sorted(set(r["run"].replace(" | uniform-mean ablation", "") for r in uniform_ablation))
        for bn in base_names:
            pts_a = [r for r in detail_rows if r["run"] == bn]
            pts_u = [r for r in detail_rows if r["run"] == f"{bn} | uniform-mean ablation"]
            if not pts_a or not pts_u:
                continue

            fig, ax = plt.subplots(1, 1, figsize=(6, 5), dpi=120)
            ax.scatter([float(r["rate_proxy"]) for r in pts_a], [float(r["teacher_drop"]) for r in pts_a], alpha=0.75, label=f"{bn} (adaptive)")
            ax.scatter([float(r["rate_proxy"]) for r in pts_u], [float(r["teacher_drop"]) for r in pts_u], alpha=0.75, label=f"{bn} (uniform-mean)")
            ax.set_title("Adaptive vs Uniform-Mean Ablation")
            ax.set_xlabel("Rate proxy")
            ax.set_ylabel("Teacher drop")
            ax.axhline(0.0, color="black", linewidth=1)
            ax.grid(alpha=0.3)
            ax.legend(fontsize=8)
            plt.tight_layout()
            plt.show()
else:
    print("Skip plotting quick-eval results.")



## Interpretation Checklist (Paper/Meeting Friendly)

Internal labels used in code:
- Stage0 = Uniform baseline (ROI-unaware)
- Stage1 = Adaptive ROI compression (no distillation)
- Stage2 = Adaptive ROI compression + distillation

How to read this notebook:
1. Training curves (grouped by Stage0/1/2) show convergence trend, not downstream detection quality.
2. Cell 9 ROI metrics are **single-frame diagnostics** (not dataset-level KPI).
3. If `TP=0, FP=0, FN>0` at threshold 0.5, check printed importance stats:
   - if `max_importance < 0.5`, empty prediction is expected by thresholding rule.
4. Cell 11/12 quick-eval is a 16-frame proxy benchmark for trend checking only.
5. For thesis tables, use this notebook for visualization + sanity checks, then run full detector evaluation separately.

